# 12b: Model-factor ablations

Design decisions
- One model factor at a time against the frozen XGBoost (depth 4, 100 trees, max_bin 32, min_child_weight 50, gamma 2): model family (LightGBM, random forest, closest equivalent settings), depth 3, 5, 6, 25 and 50 trees, max_bin 16, 64, 256, and clinically signed monotone constraints. Each variant is trained like the frozen model: training portion, seed 42, Youden threshold on 5-fold out-of-fold scores. Refitting the frozen settings this way must reproduce the frozen model hash.
- Every variant and the frozen model as reference go through the same pipeline on the same patients: prior from the same 500 training patients (mixture, lambda as frozen), candidate sets (M, order seed and node budget as frozen) for the same 500 explained test patients, certificates on the full calibration split 1, the second exact check on releasable boxes, release at the frozen targets alpha_0 = 0.05 and alpha_1 = 0.70. The policy test on calibration split 2 is not run for variants.
- Reported per variant: test AUROC, number of thresholds, exported-against-native disagreements, share of the 500 patients explained per kind with 95% intervals, test error, explanation size, calibration support, share of explanations whose box has test error above its target, median seconds per patient, and search-budget exhaustions.
- Model seeds are not varied: the frozen XGBoost uses no row or column sampling, so every seed gives the identical model (checked below).
- Candidate sets are saved on Drive every 200 patients and every finished variant is saved, so an interrupted run resumes.

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
import subprocess, sys, warnings


def quiet_warnings():
    # imodels switches on every DeprecationWarning; Colab's message code then warns on every output line
    warnings.filterwarnings('ignore', category=DeprecationWarning, module='jupyter_client')
    warnings.filterwarnings('ignore', message=r'datetime\.datetime\.utcnow')


quiet_warnings()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'z3-solver', 'lightgbm'], check=True)
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
DRIVE_ROOT = '/content/drive/MyDrive/SCAX_Research'
REPO_DIR = f'{DRIVE_ROOT}/scax-research'
for name in ('.gitconfig', '.git-credentials'):
    if os.path.exists(f'{DRIVE_ROOT}/{name}'):
        shutil.copy(f'{DRIVE_ROOT}/{name}', os.path.expanduser(f'~/{name}'))
os.chdir(REPO_DIR)
os.environ['SCAX_DRIVE_ROOT'] = DRIVE_ROOT
sys.path.insert(0, f'{REPO_DIR}/src')
pull = subprocess.run(['git', 'pull', 'origin', 'main'], capture_output=True, text=True)
print(pull.stdout.strip() or pull.stderr.strip())

In [ ]:
import json, time
import numpy as np
import pandas as pd
from scax import paths
from scax.utils import seed_everything, sha256_file, write_json
from scax.data import public, splits
from scax.certify.clopper_pearson import cp_lower, cp_upper
from scax.experiments.main_results import FrozenTask
from scax.models import train
from scax.models.export import export_model
try:
    from scax.experiments.model_ablations import ModelVariantRun, exported_vs_native, fit_variant, variants
except ImportError:
    raise RuntimeError('The stage 7 source is not loaded. Run the stage 7 batch, then Runtime > Disconnect and delete runtime, then run again.')

SEED = 42
seed_everything(SEED)
TASKS = ['cdc_diabetes', 'brfss2015_heart']
NAMES = {'cdc_diabetes': 'CDC Diabetes', 'brfss2015_heart': 'BRFSS Heart disease'}
KIND = {0: 'negative outcome', 1: 'enrichment'}
REC05 = json.load(open(f'{paths.RESULTS}/05_frozen.json'))
FT = {t: FrozenTask(t, REPO_DIR, DRIVE_ROOT, REC05['frozen_config_sha256'][t]) for t in TASKS}
CERT_DIR = paths.DERIVED / 'certificates'
MOD_DIR = paths.DERIVED / 'ablations' / 'models'; MOD_DIR.mkdir(parents=True, exist_ok=True)
N_PRIOR, N_TEST = 500, 500


def load_task(task):
    X, y = public.LOADERS[task](paths.RAW)
    code = np.load(paths.DERIVED / 'splits' / f'{task}.npy')
    assert splits.summary(code)['sha256'] == FT[task].cfg['split_sha256'], 'split differs from the frozen one'
    return X.to_numpy(dtype=float), y, code


for t in TASKS:
    print(t, 'frozen configuration', FT[t].cfg_sha[:16], '| params', FT[t].cfg['model']['params'])

## Frozen settings refitted, and seed check

In [ ]:
for task in TASKS:
    ft = FT[task]; Xa, y, code = load_task(task)
    tr = splits.indices(code, 'train')
    _, ens, _ = fit_variant('xgboost', Xa[tr], y[tr], ft.cfg['model']['params'], SEED, list(ft.ens.feature_names))
    assert ens.hash() == ft.cfg['model']['sha256'], f'{task}: refitting the frozen settings does not reproduce the frozen model'
    hashes = {s: export_model(train.fit('xgboost', Xa[tr], y[tr], ft.cfg['model']['params'], s), 'xgboost',
                              list(ft.ens.feature_names)).hash() for s in (42, 43, 44)}
    print(f'{task}: frozen model reproduced; seeds 42, 43, 44 give {len(set(hashes.values()))} distinct model(s)')

## Variants (each saved on Drive when finished)

In [ ]:
ROWS, INFO = [], []
for ti, task in enumerate(TASKS):
    ft = FT[task]; names = list(ft.ens.feature_names)
    Xa, y, code = load_task(task)
    tr = splits.indices(code, 'train')
    rng = np.random.default_rng(SEED + 120 + ti)
    prior_rows = np.sort(rng.choice(tr, N_PRIOR, replace=False))
    test_rows = np.sort(rng.choice(ft.test_rows, N_TEST, replace=False))
    alpha = {0: ft.targets['alpha_0'], 1: ft.targets['alpha_1']}
    c = ft.cfg['candidates']
    runner = ModelVariantRun(Xa, y, code, prior_rows, test_rows, c['M'], c['order_seed'], c['node_budget'], ft.lam, alpha,
                             ft.targets['delta_1'])
    todo = [('frozen model', 'frozen (XGBoost, depth 4, 100 trees, max_bin 32)', 'xgboost', ft.cfg['model']['params'])]
    todo += variants(ft.cfg['model']['params'], names)
    for factor, level, family, params in todo:
        slug = level.split(' (')[0].replace(' ', '_').replace(',', '')
        res_file = MOD_DIR / f'{task}__{slug}.json'
        if res_file.exists():
            r = json.load(open(res_file)); ROWS += r['rows']; INFO.append(r['info']); print(f'{task} | {level}: loaded from Drive'); continue
        t0 = time.time()
        if factor == 'frozen model':
            pred, disagree = ft.pred, 0
            preload = CERT_DIR / f'{task}__candidates.pkl'
        else:
            model, ens, pred = fit_variant(family, Xa[tr], y[tr], params, SEED, names)
            disagree = exported_vs_native(model, family, pred, Xa[splits.indices(code, 'test')])
            preload = None
        assert disagree == 0, f'{task} | {level}: exported and native models disagree on {disagree} test rows'
        rows, info, yhat = runner.run(pred, cache_path=MOD_DIR / f'{task}__{slug}__candidates.pkl', preload=preload,
                                      log=lambda m: print(m, flush=True))
        changed = int((yhat != ft.pred.predict(Xa[test_rows])).sum())
        info = {'task': task, 'factor': factor, 'level': level, 'family': family, 'exported_vs_native_disagreements': disagree,
                'predictions_changed_vs_frozen': changed, 'seconds_total': time.time() - t0, **info}
        rows = [{'task': task, 'factor': factor, 'level': level, **r} for r in rows]
        write_json(res_file, {'rows': rows, 'info': info})
        ROWS += rows; INFO.append(info)
        print(f"{task} | {level}: AUROC {info['auroc_test']:.4f}, explained {rows[0]['coverage']:.1%} / {rows[1]['coverage']:.1%}, "
              f"{info['median_seconds_per_cell']:.2f}s per patient, {info['seconds_total']:.0f}s", flush=True)

In [ ]:
R = pd.DataFrame(ROWS); I = pd.DataFrame(INFO)
R['kind'] = R['kind'].map(KIND)
R['explained_lo'] = [float(cp_lower(a, N_TEST, 0.05)) for a in R.accepted]
R['explained_hi'] = [float(cp_upper(a, N_TEST, 0.05)) for a in R.accepted]
R.to_csv(f'{paths.RESULTS}/12b_model_ablations.csv', index=False)
I.to_csv(f'{paths.RESULTS}/12b_model_variants.csv', index=False)
print(I[['task', 'level', 'auroc_test', 'thresholds', 'predictions_changed_vs_frozen', 'releasable_boxes', 'second_check_failures',
         'search_unknown', 'median_seconds_per_cell']].round(4).to_string(index=False))
for (task, kind), g in R.groupby(['task', 'kind'], sort=False):
    print(f'\n{NAMES[task]}, {kind} claims ({N_TEST} explained test patients)')
    print(g[['factor', 'level', 'coverage', 'explained_lo', 'explained_hi', 'error_rate', 'mean_size', 'median_support',
             'above_target_share']].round(4).to_string(index=False))

## Figure

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 8, 'axes.spines.top': False, 'axes.spines.right': False})
COL = {'cdc_diabetes': ('#0072B2', 'o'), 'brfss2015_heart': ('#D55E00', 's')}
levels = list(dict.fromkeys(R.level))
def display(l):
    if l.startswith('frozen'):
        return 'Frozen model (XGBoost, depth 4)'
    if l.startswith('max_bin'):
        return f"{l.split()[1]} threshold bins"
    return {'clinically signed': 'Monotone constraints', 'random forest': 'Random forest'}.get(l, l[0].upper() + l[1:])
label = {l: display(l) for l in levels}
fig, axes = plt.subplots(1, 2, figsize=(7.4, 4.6), sharey=True)
for j, kind in enumerate(KIND.values()):
    ax = axes[j]
    for k, task in enumerate(TASKS):
        g = R[(R.task == task) & (R.kind == kind)].set_index('level').reindex(levels)
        yv = np.arange(len(levels))[::-1] + (0.15 if k == 0 else -0.15)
        col, mk = COL[task]
        ax.errorbar(g.coverage, yv, xerr=[g.coverage - g.explained_lo, g.explained_hi - g.coverage], fmt=mk, color=col,
                    ms=4.5, lw=1, capsize=0, label=NAMES[task])
        ref = g.loc[levels[0], 'coverage']
        ax.axvline(ref, color=col, lw=0.6, ls=':')
    ax.set_yticks(np.arange(len(levels))[::-1]); ax.set_yticklabels([label[l] for l in levels])
    ax.set_title(f'{kind.capitalize()} claims', fontsize=9)
    ax.xaxis.set_major_formatter(PercentFormatter(1.0, decimals=0))
    ax.grid(axis='x', color='0.9', lw=0.6)
fig.supxlabel(f'Share of {N_TEST} explained test patients given an explanation (dotted: frozen model)', fontsize=8.5)
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc='upper center', ncol=2, frameon=False, bbox_to_anchor=(0.6, 1.02))
fig.tight_layout(rect=(0, 0, 1, 0.95))
for ext in ('png', 'pdf'):
    fig.savefig(paths.FIGURES / f'fig_ablations_model_factors.{ext}', dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
write_json(f'{paths.RESULTS}/12b_model_ablations.json', {
    'seed': SEED, 'frozen_config_sha256': {t: FT[t].cfg_sha for t in TASKS}, 'prior_patients': N_PRIOR, 'test_patients': N_TEST,
    'variants': I.to_dict('records')})
print('written: results/12b_model_ablations.csv, 12b_model_variants.csv, 12b_model_ablations.json, figures/fig_ablations_model_factors')

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
r = subprocess.run(["python", "tools/commit_cell.py",
  "12b: model-factor ablations (family, depth, trees, threshold bins, monotone constraints) for tasks B and A, figure"],
  capture_output=True, text=True)
print(r.stdout); print(r.stderr)